In [23]:
import pandas as pd
import numpy as np

# PREPEARING SALES TABLE
df_sales = pd.read_excel('../data/raw/sales_logs.xlsm', skiprows = 1, dtype={'ID transakcji  ': str})


rename_columns_dict = {'ID transakcji': 'transaction_id',
                       'Czas rozliczenia': 'transaction_time',
                       'Informacja o wyborze': 'beverage_name',
                       'Wartość rozliczenia': 'price',
                       'Payment Method (Source)': 'payment_method',
                       'Marka karty': 'card_brand'}

#cleaning and renaming columns
df_sales.columns = df_sales.columns.str.strip() #clean column names from extra spaces
df_sales = df_sales.rename(columns = rename_columns_dict) #rename necessary columns
df_sales = df_sales.drop(columns=df_sales.columns.difference(rename_columns_dict.values())) #drop unnecessary columns
df_sales = df_sales.dropna(subset=['beverage_name', 'price']) #delete rows with NaN values in specified rows
df_sales['transaction_time'] = pd.to_datetime(df_sales['transaction_time'], dayfirst=True) #change column data type
price_mask = df_sales['price'] > 0 #creatin boolean mask for price > 0 
df_sales = df_sales[price_mask] #dataframe filtering 

#spliting timestamp into different columns
df_sales['month'] = df_sales['transaction_time'].dt.month_name()
df_sales['day_of_week'] = df_sales['transaction_time'].dt.day_name()
df_sales['hour'] = df_sales['transaction_time'].dt.hour
df_sales['date'] = df_sales['transaction_time'].dt.date

#creating dictionary to standartize beverage names
beverage_mapping = {
    'kawa z mlekiem (3  7.00)\n': 'Coffee with Milk',
    'Kawa Czarna(5  6.00)\n': 'Black Coffee',
    'Moccacino(4  7.00)\n': 'Moccacino',
    'Espresso(1  6.00)\n': 'Espresso',
    'Cappuccino(2  8.00)\n': 'Cappuccino',
    'Czekolada(7  7.00)\n': 'Hot Chocolate',
    'MATCHA(8  8.49)\n': 'Cappuccino Hazelnut',
    'Latte(6  8.00)\n': 'Latte',
    'Cappuccino krem (8  8.49)\n': 'Cappuccino Hazelnut',
    'kawa z czekoladą(3  7.00)\n': 'Moccacino',
    'czarna kawa z mlekiem(3  7.00)\n': 'Coffee with Milk',
    'Kawa z mlekiem(3  7.00)\n': 'Coffee with Milk',
    'MATCHA (8  8.49)\n': 'Cappuccino Hazelnut',
    'ESPRESSO (1  6.00)\n': 'Espresso',
    'Cappuccino (2  8.00)\n': 'Cappuccino',
    'CZEKOLADA (7  7.00)\n': 'Hot Chocolate',
    'MATCHA (8  5.49)\n': 'Lemon Tea',
    'kawa z mlekiem (3  6.49)\n': 'Coffee with Milk',
    'Cappuccino Orzech łaskawy (8  5.49)\n': 'Lemon Tea',
    'hot chocolate with milk (7  7.00)\n': 'Hot Chocolate',
    'espresso (1  6.00)\n': 'Espresso',
    'kawa czarna(5  6.00)\n': 'Black Coffee',
    'Herbata(8  5.49)\n': 'Lemon Tea',
    'kawa z mlekiem (3  6.00)\n': 'Coffee with Milk',
    'Lemon tea(8  5.49)\n': 'Lemon Tea',
    'Herbata (8  5.49)\n': 'Lemon Tea'
}
df_sales['beverage_name'] = df_sales['beverage_name'].replace(beverage_mapping)

# COGS dictionary
cogs_dict = {
    'Espresso': 1.22,
    'Black Coffee': 2.24,
    'Coffee with Milk': 2.46,
    'Cappuccino': 2.67,
    'Latte': 2.67,
    'Hot Chocolate': 2.42,
    'Moccacino': 3.30,
    'Cappuccino Hazelnut': 2.80,
    'Lemon Tea': 1
}

# creating new columns of df
df_sales['cogs'] = df_sales['beverage_name'].map(cogs_dict)
df_sales['transaction_fee'] = (df_sales['price'] * 0.025).round(2)
df_sales['gross_profit_per_cup'] = df_sales['price'] - df_sales['cogs'] - df_sales['transaction_fee']

#export cleaned table to .csv
df_sales.to_csv('../data/processed/clean_sales_2025.csv', index=False)

In [31]:
# PREPEARING FIXED EXPENSES TABLE
df_fix_exp = pd.read_csv('../data/raw/fixed_expenses.csv')

#change columns data types
df_fix_exp['month'] = pd.to_datetime(df_fix_exp['month'], format='%m/%Y').dt.strftime('%B') # str to month name (like 'January')

conv_col = df_fix_exp.columns.difference(['month']) #all columns exept month

# str to float round 2
for col_name in df_fix_exp[conv_col].columns: 
    df_fix_exp[col_name] = (
        df_fix_exp[col_name]
            .str.replace('zł', '', case = False)
            .str.strip()
            .astype(float)
            .round(2)
    )

df_fix_exp['zus'] = 420.00 #add ZUS fee into expenses

# #export cleaned table to .csv
df_fix_exp.to_csv('../data/processed/clean_fixed_expenses_2025.csv', index=False)